In [13]:
import joblib

svm_model = joblib.load("svm_model.pkl")
scaler = joblib.load("scaler.pkl")

In [15]:
import tkinter as tk
from tkinter import messagebox
import joblib


# =========================================================
# LOAD SAVED MODEL
# =========================================================

svm_model = joblib.load("svm_model.pkl")
scaler = joblib.load("scaler.pkl")


# =========================================================
# COLORS
# =========================================================

BG_COLOR = "#F4F8F2"
CARD_COLOR = "#FFFFFF"
PRIMARY = "#2E7D32"
PRIMARY_DARK = "#1B5E20"
ACCENT = "#81C784"
TEXT_COLOR = "#263238"
MUTED = "#607D6B"
RESULT_BG = "#E8F5E9"


# =========================================================
# MAIN WINDOW
# =========================================================

window = tk.Tk()
window.title("Crop Recommendation System")
window.geometry("760x720")
window.configure(bg=BG_COLOR)
window.resizable(False, False)


# =========================================================
# HEADER
# =========================================================

header = tk.Frame(
    window,
    bg=PRIMARY,
    height=110
)
header.pack(fill="x")
header.pack_propagate(False)

tk.Label(
    header,
    text="🌱 Crop Recommendation System",
    font=("Arial", 24, "bold"),
    bg=PRIMARY,
    fg="white"
).pack(pady=(22, 5))

tk.Label(
    header,
    text="Get a suitable crop recommendation based on soil and weather conditions",
    font=("Arial", 11),
    bg=PRIMARY,
    fg="#E8F5E9"
).pack()


# =========================================================
# INPUT CARD
# =========================================================

input_card = tk.Frame(
    window,
    bg=CARD_COLOR,
    padx=30,
    pady=25
)

input_card.pack(
    padx=40,
    pady=25,
    fill="x"
)


tk.Label(
    input_card,
    text="Enter Soil & Weather Parameters",
    font=("Arial", 15, "bold"),
    bg=CARD_COLOR,
    fg=TEXT_COLOR
).grid(
    row=0,
    column=0,
    columnspan=4,
    pady=(0, 20)
)


# =========================================================
# INPUT FIELDS
# =========================================================

entries = {}


def create_input(label_text, row, column, variable_name):

    tk.Label(
        input_card,
        text=label_text,
        font=("Arial", 10, "bold"),
        bg=CARD_COLOR,
        fg=TEXT_COLOR
    ).grid(
        row=row,
        column=column,
        sticky="w",
        padx=10,
        pady=(5, 2)
    )

    entry = tk.Entry(
        input_card,
        font=("Arial", 11),
        width=20,
        relief="solid",
        bd=1
    )

    entry.grid(
        row=row + 1,
        column=column,
        padx=10,
        pady=(0, 12)
    )

    entries[variable_name] = entry


create_input("Nitrogen (N)", 1, 0, "N")
create_input("Phosphorus (P)", 1, 1, "P")

create_input("Potassium (K)", 3, 0, "K")
create_input("Temperature (°C)", 3, 1, "temperature")

create_input("Humidity (%)", 1, 2, "humidity")
create_input("Soil pH", 3, 2, "ph")

create_input("Rainfall (mm)", 1, 3, "rainfall")


# =========================================================
# RESULT CARD
# =========================================================

result_card = tk.Frame(
    window,
    bg=RESULT_BG,
    padx=25,
    pady=20
)

result_card.pack(
    padx=40,
    pady=(0, 20),
    fill="x"
)


tk.Label(
    result_card,
    text="Recommended Crop",
    font=("Arial", 12, "bold"),
    bg=RESULT_BG,
    fg=MUTED
).pack()


crop_result = tk.Label(
    result_card,
    text="—",
    font=("Arial", 24, "bold"),
    bg=RESULT_BG,
    fg=PRIMARY_DARK
)

crop_result.pack(pady=(5, 15))


tk.Label(
    result_card,
    text="Fertilizer Recommendation",
    font=("Arial", 11, "bold"),
    bg=RESULT_BG,
    fg=MUTED
).pack()


fertilizer_result = tk.Label(
    result_card,
    text="—",
    font=("Arial", 12),
    bg=RESULT_BG,
    fg=TEXT_COLOR
)

fertilizer_result.pack(pady=(5, 0))


# =========================================================
# FERTILIZER LOGIC
# =========================================================

def fertilizer_recommendation(N, P, K):

    # Simple rule-based recommendation
    # based on the nutrient that is relatively low.

    if N < P and N < K:
        return "Nitrogen-rich fertilizer is recommended."

    elif P < N and P < K:
        return "Phosphorus-rich fertilizer is recommended."

    elif K < N and K < P:
        return "Potassium-rich fertilizer is recommended."

    else:
        return "Balanced NPK fertilizer is recommended."


# =========================================================
# PREDICTION FUNCTION
# =========================================================

def predict_crop():

    try:

        N = float(entries["N"].get())
        P = float(entries["P"].get())
        K = float(entries["K"].get())

        temperature = float(
            entries["temperature"].get()
        )

        humidity = float(
            entries["humidity"].get()
        )

        ph = float(
            entries["ph"].get()
        )

        rainfall = float(
            entries["rainfall"].get()
        )

        # Check pH range
        if ph < 0 or ph > 14:
            messagebox.showerror(
                "Invalid Input",
                "Soil pH must be between 0 and 14."
            )
            return

        # Create input in the SAME order as training data
        user_input = [[
            N,
            P,
            K,
            temperature,
            humidity,
            ph,
            rainfall
        ]]

        # Scale using the SAME scaler used during training
        user_input_scaled = scaler.transform(
            user_input
        )

        # SVM prediction
        prediction = svm_model.predict(
            user_input_scaled
        )

        crop = prediction[0]

        # Display crop
        crop_result.config(
            text=crop.upper()
        )

        # Fertilizer recommendation
        fertilizer = fertilizer_recommendation(
            N, P, K
        )

        fertilizer_result.config(
            text=fertilizer
        )

    except ValueError:

        messagebox.showerror(
            "Invalid Input",
            "Please enter valid numerical values for all fields."
        )

    except Exception as e:

        messagebox.showerror(
            "Error",
            f"Something went wrong:\n{e}"
        )


# =========================================================
# RESET FUNCTION
# =========================================================

def reset_fields():

    for entry in entries.values():
        entry.delete(0, tk.END)

    crop_result.config(text="—")

    fertilizer_result.config(text="—")


# =========================================================
# BUTTONS
# =========================================================

button_frame = tk.Frame(
    window,
    bg=BG_COLOR
)

button_frame.pack(
    pady=(0, 20)
)


predict_button = tk.Button(
    button_frame,
    text="🌱  Predict Crop",
    command=predict_crop,
    font=("Arial", 11, "bold"),
    bg=PRIMARY,
    fg="white",
    activebackground=PRIMARY_DARK,
    activeforeground="white",
    relief="flat",
    padx=25,
    pady=10,
    cursor="hand2"
)

predict_button.pack(
    side="left",
    padx=8
)


reset_button = tk.Button(
    button_frame,
    text="Reset",
    command=reset_fields,
    font=("Arial", 11),
    bg="#ECEFF1",
    fg=TEXT_COLOR,
    activebackground="#CFD8DC",
    relief="flat",
    padx=25,
    pady=10,
    cursor="hand2"
)

reset_button.pack(
    side="left",
    padx=8
)


# =========================================================
# FOOTER
# =========================================================

tk.Label(
    window,
    text="Machine Learning based Crop Recommendation",
    font=("Arial", 9),
    bg=BG_COLOR,
    fg=MUTED
).pack(
    side="bottom",
    pady=8
)


# =========================================================
# START GUI
# =========================================================

window.mainloop()